# Global preparations

In [1]:
import sys
import os
import matplotlib.pyplot as plt
import time
import torch
import torchvision
import logging
import numpy as np
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.transforms.functional import to_pil_image
from PIL import Image
from tqdm.notebook import tqdm

device = 'cuda:7'
summation_dtype = torch.float32
random_seed = 1
torch.manual_seed(random_seed)
np.random.seed(random_seed)
torch.backends.cudnn.enabled = True
print(torch.__version__)

2.1.2+cu121


# Text data preparation

Let's read small fineweb fragment

In [54]:
import gdown
url = 'https://drive.google.com/file/d/1vWjyIpU6wvCPtx2OdV_4M3MXX6FsOpxV/view'
output = 'fineweb_texts.txt'
if not os.path.exists(output):
    gdown.download(url, output, quiet=False, fuzzy=True)
size = os.path.getsize(output)
if size < 233 * 1024 * 1024:
    raise RuntimeError(f'Download failed: file size {size/1024/1024:.1f} MB')

In [55]:
CONTEXT_SIZE = 32

In [56]:
from spiky.util.text_snippet_sampler import TextSnippetSampler

snippet_sampler = TextSnippetSampler('fineweb_texts.txt', CONTEXT_SIZE + 1, 10000, device)

In [57]:
snippet_sampler.sample_training_batch(2)

tensor([[117, 100, 121,  32, 111, 102,  32, 114, 101, 108, 105, 103, 105, 111,
         110,  10,  84, 104, 101, 114, 101,  32,  97, 114, 101,  32, 100, 111,
         117,  98, 116, 115,  32],
        [116, 101,  32, 111, 102,  32, 105, 100, 101, 110, 116, 105, 116, 121,
          32, 116, 104, 101, 102, 116,  32, 102, 111, 114,  32,  97, 100, 117,
         108, 116, 115,  46,  10]], device='cuda:7', dtype=torch.int32)

In [58]:
snippet_sampler.batch_to_text(snippet_sampler.sample_training_batch(4))

['ionally, public feared Sovietâ\x80\x99s',
 ' metamorphosed sediments and rock',
 'industrial trade unionism, which ',
 'aine.\nAs for the main part of Ukr']

In [59]:
for test_batch in snippet_sampler.testing_batches_iterator(4):
    print(snippet_sampler.batch_to_text(test_batch))
    break

['thereâ\x80\x99s fighting. The explosion', 'er with the UNESCO office for Ira', 'own. Hold this stretch for 30 sec', 'cessful in treating Sinusitis.\nDe']


# LUTTransformer (lutorch sketch)

Transformer sketched with **lutorch** primitives: `MultiHeadLut` and `LUTCrossAttention`.
- **Token embedder**: `nn.Embedding`
- **Per layer**: self-attention via `LUTCrossAttention` (LUT-based (Q,K) scores) + value projection with `MultiHeadLut`; then FFN with `MultiHeadLut`
- **Unembed**: `MultiHeadLut` from hidden to vocab size

In [60]:
# LUTTransformer sketch using lutorch: MultiHeadLut, LUTCrossAttention
import torch.nn as nn
from spiky.lutorch.multi_head_lut import MultiHeadLut
from spiky.lutorch.lut_cross_attention import LUTCrossAttention, PairProcessingConfig, PairProcessingMode

class LUTTransformer(nn.Module):
    """Transformer with lutorch primitives: MultiHeadLut + LUTCrossAttention."""

    def __init__(
        self,
        vocab_size=256,
        embedding_dim=32,
        context_size=32,
        num_layers=6,
        num_heads=4,
        n_anchor_pairs_attn=14,
        n_anchor_pairs_ffn=14,
        n_positional_buckets=8,
        tables_per_head_attn=32,
        tables_per_head_value=16,
        ffn_tables=16,
        dropout=0.0,
        smooth_mode=True,
        device=None
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.context_size = context_size
        self.num_heads = num_heads
        dev = device or torch.device("cpu")

        with torch.no_grad():
            self.token_embedder = nn.Embedding(vocab_size, embedding_dim, device=dev)
            # Match spike_QK: W_embed uses randn*0.1 (zero-mean Gaussian)
            self.token_embedder.weight.copy_(torch.randn(self.token_embedder.weight.shape, device=dev) * 0.1)

            self.layers = nn.ModuleList()
            for _ in range(num_layers):
                attn_lut = MultiHeadLut(
                    input_dim=embedding_dim,
                    n_heads=num_heads,
                    n_outputs=1,
                    n_anchor_pairs=n_anchor_pairs_attn,
                    tables_per_head=tables_per_head_attn,
                    n_buckets=n_positional_buckets,
                    smooth_mode=smooth_mode,
                    device=dev
                )
                attn_lut.projection.weights.copy_(
                    torch.randn(attn_lut.projection.weights.shape, device=dev) * 0.001
                )
                cross_attn = LUTCrossAttention(
                    attn_lut, 
                    causal=True,
                    attention_temperature=0.25,
                    n_positional_buckets=n_positional_buckets
                )
                value_lut = MultiHeadLut(
                    input_dim=embedding_dim,
                    n_heads=num_heads,
                    n_outputs=embedding_dim // num_heads,
                    n_anchor_pairs=n_anchor_pairs_attn,
                    tables_per_head=tables_per_head_value,
                    smooth_mode=smooth_mode,
                    device=dev,
                )
                # LProjection defaults to zeros; spike_QK V-LUT uses randn*0.001 (was missing -> value path was dead)
                value_lut.projection.weights.copy_(
                    torch.randn(value_lut.projection.weights.shape, device=dev) * 0.001
                )
                ffn_lut = MultiHeadLut(
                    input_dim=embedding_dim,
                    n_heads=1,
                    n_outputs=embedding_dim,
                    n_anchor_pairs=n_anchor_pairs_ffn,
                    tables_per_head=ffn_tables,
                    smooth_mode=smooth_mode,
                    device=dev,
                )
                ffn_lut.projection.weights.copy_(
                    torch.randn(ffn_lut.projection.weights.shape, device=dev) * 0.001
                )
                self.layers.append(nn.ModuleDict({
                    "cross_attn": cross_attn,
                    "value_lut": value_lut,
                    "attn_dropout": nn.Dropout(dropout),
                    "ffn": ffn_lut,
                    "ffn_dropout": nn.Dropout(dropout),
                }))

    def forward(self, tokens):
        # tokens: [B, S]
        B, S = tokens.shape
        with torch.no_grad():
            z = self.token_embedder(tokens)  # [B, S, E]
        for layer in self.layers:
            attn_weights = layer["cross_attn"](z, z)  # [B, S, S, H]
            v = layer["value_lut"](z.reshape(-1, self.embedding_dim))  # [B*S, H, E//H]
            v = v.reshape(B, S, self.num_heads, -1)  # [B, S, H, E//H]
            # Apply attention per head (matmul), then concat heads -> [B, S, E]
            # [B, H, S, S] @ [B, H, S, E//H] -> [B, H, S, E//H]
            attn_out = attn_weights.permute(0, 3, 1, 2) @ v.permute(0, 2, 1, 3)  # [B, H, S, E//H]
            attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, S, self.embedding_dim)  # [B, S, E]
            z = z + layer["attn_dropout"](attn_out)  # [B, S, E]
            ffn_out = layer["ffn"](z.reshape(-1, self.embedding_dim)).reshape(B, S, -1)  # [B, S, E]
            z = z + layer["ffn_dropout"](ffn_out)  # [B, S, E]
        z_norm = z / (z.norm(dim=-1, keepdim=True) + 1e-6)
        logits = z_norm @ self.token_embedder.weight.T / 0.1 # [B, S, E] @ [E, V] -> [B, S, V]
        return logits

lut_transformer = None
optimizer = None
sched = None
if device != 'cpu':
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
lut_transformer = LUTTransformer(
    context_size=CONTEXT_SIZE,
    smooth_mode=False,
    device=device
)
print(lut_transformer)

LUTTransformer(
  (token_embedder): Embedding(256, 32)
  (layers): ModuleList(
    (0-5): 6 x ModuleDict(
      (cross_attn): LUTCrossAttention(
        (multi_head_lut): MultiHeadLut(
          (lookup): AnchorPairsLookup()
          (projection): LProjection()
        )
      )
      (value_lut): MultiHeadLut(
        (lookup): AnchorPairsLookup()
        (projection): LProjection()
      )
      (attn_dropout): Dropout(p=0.0, inplace=False)
      (ffn): MultiHeadLut(
        (lookup): AnchorPairsLookup()
        (projection): LProjection()
      )
      (ffn_dropout): Dropout(p=0.0, inplace=False)
    )
  )
)


In [61]:
total = sum(p.numel() for p in lut_transformer.parameters())
trainable = sum(p.numel() for p in lut_transformer.parameters() if p.requires_grad)

print("total:", total)
print("trainable:", trainable)
print("frozen:", total - trainable)

total: 201334784
trainable: 201334784
frozen: 0


In [62]:
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

SCALE = 5.0
MAX_RATE = 0.01
def lr_func(t):
    return min(MAX_RATE, SCALE / (1 + t)**0.5)

print(f"Crossover point for LR: {(SCALE / MAX_RATE )**2:,}")

lr = 1.0
optimizer = optim.SGD(lut_transformer.layers.parameters(), lr=lr)
# Match spike_QK: embedder uses global LR scale (0.01); 0.001 was 10x too small
optimizer_embedder = optim.Adam(lut_transformer.token_embedder.parameters(), lr=0.01)

steps=1000000
# sched = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=steps
# )
# sched = None
sched = LambdaLR(optimizer, lr_lambda=lr_func)
# sched_embedder = LambdaLR(optimizer_embedder, lr_lambda=lr_func)
# LUTTransformerLutorch has no set_external_learning_rate_hook

Crossover point for LR: 250,000.0


In [63]:
lut_train_losses = []
lut_test_losses = []

In [69]:
import torch
import torch.nn.functional as F

test_batch_size = 128

def generate_text_lut(lut_model, prefix, length, device):
    ctx = list(prefix.encode("utf-8"))
    ctx = ctx[-CONTEXT_SIZE:]

    for _ in range(length):
        x = torch.zeros([CONTEXT_SIZE], dtype=torch.long, device=device).unsqueeze(0)
        trunc_ctx = ctx[-CONTEXT_SIZE:]
        x[0, -len(trunc_ctx):] = torch.tensor(trunc_ctx, dtype=torch.long, device=device) 
        logits = lut_model(x)
        probs = torch.softmax(logits[:,-1,:], dim=-1)[0]
        next_id = torch.multinomial(probs, 1).item()
        ctx.append(next_id)

    ctx_safe = [c if c != 0 else 32 for c in ctx]
        
    return bytes(ctx_safe).decode("latin1", errors="ignore")

def evaluate_model(model, sampler, B, last_position_only=False):
    """last_position_only=True matches spike_QK (loss only at last token; expect ~6.48 untrained)."""
    model.eval()
    losses = []
    device = next(model.parameters()).device

    with torch.no_grad():
        for batch in sampler.testing_batches_iterator(B):   # [B, C]
            inp = batch[:, :-1]          # [B, C-1]
            tgt = batch[:, 1:].long()    # [B, C-1]

            logits = model(inp)   # [B, C-1, 256]

            B_, T, V = logits.shape
            if last_position_only:
                loss = F.cross_entropy(logits[:, -1, :], tgt[:, -1], reduction='mean')
                losses.append(loss.item())
            else:
                loss = F.cross_entropy(
                    logits.reshape(B_ * T, V),
                    tgt.reshape(B_ * T),
                    reduction='none'
                ).sum()
                losses.append(loss.item() / (CONTEXT_SIZE * B))

    # ---- small generation demo ----
    prefix = "Once upon a time "
    gen = generate_text_lut(model, prefix, length=80, device=device)
    print("\n[GEN]:", gen, "\n")

    model.train()
    return sum(losses) / len(losses) #  / (CONTEXT_SIZE * B)

In [70]:
# Use last_position_only=True to match spike_QK (expect ~6.48 untrained). False = mean over all positions (~5.4).
test_loss = evaluate_model(lut_transformer, snippet_sampler, test_batch_size, last_position_only=True)
test_loss


[GEN]: Once upon a time ECQÓ+cn9¦÷6aaÁJ5	þrnÂ@I,²ÇÍKem
ÎK½ÙÉuYöÔÀ

S®$F¸0
á-ô
÷@nÉJS1ª 



5.444685253915908

In [38]:
test_every=1000
train_loss = None
alpha = 0.01
batch_size = 128

pbar = tqdm(total=steps)
lut_transformer.train()

for step in range(0, steps + 1):
    x = snippet_sampler.sample_training_batch(batch_size)   # [B, C]
    inp = x[:, :-1]                                         # [B, C-1]
    tgt = x[:, 1:].long()                                   # [B, C-1]

    logits = lut_transformer(inp)      # [B, C-1, 256]

    # Match spike_QK: train on last position only (reduction='mean') so objective and gradient scale match
    loss = F.cross_entropy(logits[:, -1, :], tgt[:, -1], reduction='mean')

    optimizer.zero_grad()
    optimizer_embedder.zero_grad()
    loss.backward()
    optimizer.step()
    optimizer_embedder.step()
    if sched is not None:
        for _ in range(x.shape[0]):
            sched.step()
            # sched_embedder.step()
        # sched.step()
        # sched_embeddder.step()

    loss_value = loss.item()  # already mean over B (last position only)
    train_loss = loss_value if train_loss is None else (1 - alpha) * train_loss + alpha * loss_value
    pbar.update(1)
    if step % 10 == 0:
        pbar.set_description(f"loss={train_loss:.4f}, lr {lr if sched is None else sched.get_last_lr()[0]:.8f}")

    if step % test_every == 0:
        test_loss = evaluate_model(lut_transformer, snippet_sampler, test_batch_size, last_position_only=True)
        if len(lut_train_losses) == 0 or step > 0:
            lut_train_losses.append(train_loss)
            lut_test_losses.append(test_loss)
        print(f"[TEST] step {step}: loss={test_loss:.4f}")
#         if step > 0 and batch_size < 384:
#             print(f"batch_size {batch_size} -> {batch_size + 32}")
#             batch_size += 32

  0%|          | 0/1000000 [00:00<?, ?it/s]


[GEN]: Once upon a time |<#Ñµø[fm\d½íY;8á;¢j6Z3xðvªp ­øÕþ¿Òw~ab¡
XÞ¾Ð´ÉËIîµ8²¢^Ìÿ±V| 

[TEST] step 0: loss=5.2086

[GEN]: Once upon a time and exteti 15 votality ats belia, butant warÞy sciet anal outave leprable hanan  

[TEST] step 1000: loss=1.9690

[GEN]: Once upon a time use/folling Hach 1hal and 3 of this abilition, Devit C
Yow Colly, the Parch 12 

[TEST] step 2000: loss=1.8720

[GEN]: Once upon a time so; cha Kizcomes two GSSH/2inible Read Gas-body composing it his ately.
Offity m 

[TEST] step 3000: loss=1.8002


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# assume train_losses and test_losses are Python lists of equal length

steps = [i * 1000 for i in range(len(lut_train_losses))]

plt.figure(figsize=(6,4))
plt.plot(steps, lut_train_losses, label="train")
plt.plot(steps, lut_test_losses, label="test")
plt.ylim(top=2.0)
plt.xlabel("steps")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [65]:
test_loss

5.505621819556514

In [14]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

profile_steps = 200

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    for step in range(profile_steps):

        with record_function("sample_batch"):
            x = snippet_sampler.sample_training_batch(batch_size)
            inp = x[:, :-1]
            tgt = x[:, 1:].long()

        with record_function("forward"):
            logits = lut_transformer(inp)

        with record_function("loss"):
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, V),
                tgt.reshape(B * T),
                reduction="none"
            ).sum()

        with record_function("backward+step"):
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        with record_function("scheduler"):
            if sched is not None:
                for _ in range(x.shape[0]):
                    sched.step()

        prof.step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=40))
prof.export_chrome_trace("trace.json")

STAGE:2026-03-05 01:41:34 304266:304266 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-03-05 01:42:11 304266:304266 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-03-05 01:42:12 304266:304266 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         1.52%     757.733ms        25.73%       12.834s      64.170ms       0.000us         0.00%       20.728s     103.642ms      18.75 Kb      18.75 Kb     713.63 Gb    -741.48 G

In [26]:
steps=1000000


In [17]:
from spiky.lutorch.lut_helpers import logarithmic_pe_buckets, rpe_matrix

In [29]:
pe_buckets = logarithmic_pe_buckets(8, 32, device)

In [30]:
pe_buckets

tensor([0, 1, 2, 3, 4, 4, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7], device='cuda:7')

In [32]:
rpe_matrix(pe_buckets, 32, device).T

tensor([[0, 1, 2,  ..., 7, 7, 7],
        [0, 0, 1,  ..., 7, 7, 7],
        [0, 0, 0,  ..., 7, 7, 7],
        ...,
        [0, 0, 0,  ..., 0, 1, 2],
        [0, 0, 0,  ..., 0, 0, 1],
        [0, 0, 0,  ..., 0, 0, 0]], device='cuda:7')

In [36]:
causal_mask = torch.tril(torch.ones(32, 32, device=device), diagonal=-1).unsqueeze(0).unsqueeze(-1)
causal_mask == 0

tensor([[[[ True],
          [ True],
          [ True],
          ...,
          [ True],
          [ True],
          [ True]],

         [[False],
          [ True],
          [ True],
          ...,
          [ True],
          [ True],
          [ True]],

         [[False],
          [False],
          [ True],
          ...,
          [ True],
          [ True],
          [ True]],

         ...,

         [[False],
          [False],
          [False],
          ...,
          [ True],
          [ True],
          [ True]],

         [[False],
          [False],
          [False],
          ...,
          [False],
          [ True],
          [ True]],

         [[False],
          [False],
          [False],
          ...,
          [False],
          [False],
          [ True]]]], device='cuda:7')

In [37]:
attention_scores = torch.zeros([2, 32, 32, 1], device=device)

In [38]:
attention_scores = attention_scores.masked_fill(causal_mask == 0, float('-inf')).squeeze(3)    

In [40]:
F.softmax(attention_scores[:, 0])

/tmp/ipykernel_367987/2957606799.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(attention_scores[:, 0])


tensor([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan]], device='cuda:7')

In [25]:
attention_scores.shape

torch.Size([2, 32, 32, 1])

In [18]:
seq_len = 32
batch_size = 3
rows_local, cols_local = torch.tril_indices(
    seq_len, seq_len, offset=-1, device=device
)  # [num_pairs_single]

offsets = torch.arange(batch_size, device=device) * seq_len  # [B]
_cached_batched_rows = (
    rows_local.unsqueeze(0) + offsets.unsqueeze(1)
).reshape(-1)  # [P], where P = B * num_pairs_single
_cached_batched_cols = (
    cols_local.unsqueeze(0).expand(batch_size, -1)
).reshape(-1)  # [P]

# Within-sequence key indices for scattering into [B*S, S, H]
_cached_key_indices = (_cached_batched_cols % seq_len).contiguous()  # [P]

pe_buckets = logarithmic_pe_buckets(8, seq_len, device)
rpe = rpe_matrix(pe_buckets, seq_len, device)  # [S, S]
rpe_pairs = rpe[rows_local, cols_local]  # [num_pairs_single]
_cached_bucket_indices = rpe_pairs.repeat(batch_size).contiguous()  # [P]

In [24]:
_cached_batched_rows[:16], _cached_batched_cols[:16]

(tensor([1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 5, 6], device='cuda:7'),
 tensor([0, 0, 1, 0, 1, 2, 0, 1, 2, 3, 0, 1, 2, 3, 4, 0], device='cuda:7'))

In [26]:
_cached_batched_rows[-16:], _cached_batched_cols[-16:]

(tensor([95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95, 95],
        device='cuda:7'),
 tensor([15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
        device='cuda:7'))

In [16]:
cols_local

tensor([ 0,  0,  1,  0,  1,  2,  0,  1,  2,  3,  0,  1,  2,  3,  4,  0,  1,  2,
         3,  4,  5,  0,  1,  2,  3,  4,  5,  6,  0,  1,  2,  3,  4,  5,  6,  7,
         0,  1,  2,  3,  4,  5,  6,  7,  8,  0,  1,  2,  3,  4,  5,  6,  7,  8,
         9,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,
        12,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,  0,  1,  2,
         3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,
         8,  9, 10, 11, 12, 13, 14, 15, 16,  0,  1,  2,  3,  4,  5,  6,  7,  8,
         9, 10, 11, 12, 13, 14, 15, 16, 17,  0,  1,  2,  3,  4,  5,  6,  7,  8,
         9, 10, 11, 12, 13, 14, 15, 16, 17, 18,  0,  1,  2,  3,  4,  5,  6,  7,
         8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11, 12, 13, 